# Clase 13: Optimización de Rendimiento y Cierre del Bloque 2

### Machine Learning (RIA-805) — Bloque 2 — Factibilidad Técnico-Financiera

**Docente:** Marco Capio  
**Fecha:** Semana 3 del bloque comprimido

**Objetivo:** Integrar todas las técnicas del Bloque 2 en un pipeline completo de evaluación y optimización, aplicando threshold tuning, class_weight, feature engineering, feature selection y Optuna para minimizar el costo total.

**Conexión con la práctica:** En P2-11 y P2-12 documentarán y entregarán su pipeline completo. Hoy verán exactamente cómo se integra todo.

## Repaso del Bloque 2

| Clase | Tema | ¿Qué pregunta responde? |
|---|---|---|
| 7 | Factibilidad | ¿Tiene sentido técnica y económicamente resolver este problema? |
| 8 | Métricas | ¿Cómo medimos el rendimiento cuando las clases están desbalanceadas? |
| 9 | Costos y umbral | ¿Cuánto cuestan los errores y qué umbral de decisión conviene? |
| 10 | Validación cruzada y Optuna | ¿Qué hiperparámetros minimizan el costo de forma robusta? |
| 11 | Fundamentos de aprendizaje por refuerzo | ¿Cómo se modelan las decisiones secuenciales? |
| 12 | Entrenamiento de RL | ¿Cómo entrenamos y evaluamos un agente? |

Hoy integramos todo en un pipeline de 7 etapas.

## El pipeline completo

```text
┌──────────────────────┐
│ 1. Problema y costos │ → 2. Modelo base → 3. CV
└──────────────────────┘
          ↓
4. Matriz de costos → 5. Threshold → 6. Optuna → 7. Documentación
```

1. **Problema y costos:** precisar la clase positiva y el impacto económico de cada error.
2. **Modelo base:** establecer una referencia sencilla y reproducible.
3. **Validación cruzada (CV):** estimar la variabilidad del rendimiento solo con entrenamiento.
4. **Matriz de costos:** traducir falsos positivos y negativos a una misma unidad de negocio.
5. **Threshold:** ajustar la regla de decisión usando validación, no el conjunto final de prueba.
6. **Optuna:** explorar hiperparámetros con una función objetivo alineada al costo.
7. **Documentación:** registrar resultados, configuración y límites del modelo.

El notebook también compara técnicas complementarias dentro de estas etapas.

**Si el modelo no es apto después de la etapa 6, volver a la etapa 1 o 2.**

## Técnicas de mejora de rendimiento

Optuna no es la única manera de mejorar un modelo. Cada técnica introduce supuestos y puede empeorar la generalización; por ello se compara con una validación consistente.

### 4.1 Feature engineering

Consiste en crear atributos a partir de los existentes: **ratios** (por ejemplo, N/P), **interacciones** (producto de dos variables) y **transformaciones** (logaritmos o escalas). Funciona cuando la nueva representación expresa una relación útil para el dominio. El riesgo es introducir ruido, divisiones inestables o relaciones espurias; las transformaciones y sus parámetros deben aprenderse sin mirar validación o prueba.

### 4.2 Feature selection

Reduce atributos para simplificar el modelo:

- **Filtro:** selecciona con estadísticas independientes del modelo, como correlación o información mutua.
- **Wrapper:** compara subconjuntos entrenando repetidamente un modelo (por ejemplo, selección recursiva).
- **Embedded:** selecciona durante el ajuste, como coeficientes L1 o importancias de árboles.

Puede reducir ruido, costo y sobreajuste cuando hay variables irrelevantes. El riesgo es eliminar señales débiles pero útiles o seleccionar de forma optimista si se consulta el conjunto de prueba.

### 4.3 Balanceo de clases

| Técnica | Qué hace | Cuándo considerar |
|---|---|---|
| `class_weight` | Aumenta la penalización de errores en la clase minoritaria. | Buen primer intento; conserva las observaciones originales. |
| SMOTE | Interpola ejemplos sintéticos de la clase minoritaria. | Desbalance severo con variables numéricas y suficientes vecinos representativos. |
| Under-sampling | Reduce ejemplos de la clase mayoritaria. | Hay muchos ejemplos redundantes y se acepta perder información. |
| Over-sampling | Duplica o remuestrea la clase minoritaria. | La clase minoritaria tiene pocos ejemplos; controlar el riesgo de sobreajuste. |

El remuestreo se realiza exclusivamente en los pliegues de entrenamiento, nunca antes de separar prueba.

### 4.4 Ensemble methods

- **Bagging:** entrena modelos en muestras bootstrap y combina sus predicciones (Random Forest).
- **Boosting:** construye modelos secuenciales que corrigen errores anteriores.
- **Stacking:** aprende un modelo de nivel superior a partir de predicciones fuera de pliegue.
- **Voting:** combina votos de varios clasificadores mediante mayoría o probabilidades.

Los ensembles pueden mejorar robustez, pero agregan latencia, complejidad e interpretabilidad limitada.

### 4.5 Regularización

- **L1:** induce coeficientes exactamente cero y puede seleccionar variables.
- **L2:** reduce magnitudes de coeficientes sin eliminarlos.
- **Dropout:** apaga unidades aleatoriamente durante el entrenamiento de redes neuronales.
- **Early stopping:** detiene el ajuste cuando la métrica de validación deja de mejorar.

Su objetivo común es controlar el sobreajuste; la técnica aplicable depende de la familia del modelo.

## ¿Qué técnica usar primero?

Un orden práctico de experimentación es:

1. **Threshold:** cambia la decisión sin volver a entrenar el clasificador.
2. **Optuna:** busca hiperparámetros con una función objetivo explícita.
3. **`class_weight`:** compensa el desbalance durante el entrenamiento.
4. **Feature engineering:** expresa relaciones que el modelo podría no capturar fácilmente.
5. **Feature selection:** elimina atributos poco útiles.
6. **SMOTE:** prueba generación de ejemplos cuando el desbalance lo justifique.
7. **Ensemble:** combina modelos si la mejora compensa su complejidad.
8. **Regularización:** controla el sobreajuste según el tipo de modelo.

**Regla: siempre validar cada técnica con CV.** El conjunto de prueba se reserva para una evaluación final; no debe decidir qué técnica, variables o umbral elegir.

In [ ]:
# Importamos las herramientas del análisis, modelado, visualización y persistencia.
%matplotlib inline
import json
import os
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# SMOTE es opcional; el resto del notebook sigue disponible si falta imbalanced-learn.
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except ImportError:
    SMOTE = None
    ImbPipeline = None

# Fijamos el estilo, el nivel de logs y la semilla para reproducir los experimentos.
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42

# Detectamos el directorio raíz tanto si Jupyter se abrió desde la raíz como desde notebooks/.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (candidate / "data" / "processed" / "crop.parquet").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró data/processed/crop.parquet desde el directorio actual ni sus padres."
    )

# Creamos las carpetas destino que se usarán en las últimas etapas.
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
for directory in (REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if SMOTE is None:
    print("SMOTE no está disponible. Si se desea ejecutar esa técnica, instale con: %pip install imbalanced-learn")
print(f"Raíz del proyecto: {PROJECT_ROOT}")

## Etapa 1: Definir problema y costos

**Problema binario: predecir 'rice' (positivo) vs no 'rice' (negativo). Costos: C_FP=$50, C_FN=$500.**

Un falso negativo omite una recomendación de arroz que sí correspondía y cuesta diez veces más que un falso positivo. Este supuesto de costos convierte el umbral en una decisión de negocio, no solo en una preferencia estadística.

In [ ]:
# Cargamos el parquet y construimos la etiqueta positiva antes de separar los datos.
data_path = PROJECT_ROOT / "data" / "processed" / "crop.parquet"
df = pd.read_parquet(data_path)
if "label" not in df.columns:
    raise KeyError("El parquet debe incluir la columna 'label'.")

target_binario = (df["label"].astype(str).str.lower() == "rice").astype(int)
X = df.drop(columns=["label"]).copy()
y = target_binario.rename("target_binario")
C_FP, C_FN = 50, 500

# Reservamos prueba y luego una validación tomada únicamente del conjunto de entrenamiento.
X_trainval, X_test_raw, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20, stratify=y_trainval, random_state=SEED
)

# Ajustamos el escalador solo con entrenamiento para evitar fuga de información.
scaler = RobustScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw), columns=X_train_raw.columns, index=X_train_raw.index
)
X_val = pd.DataFrame(
    scaler.transform(X_val_raw), columns=X_val_raw.columns, index=X_val_raw.index
)
X_test = pd.DataFrame(
    scaler.transform(X_test_raw), columns=X_test_raw.columns, index=X_test_raw.index
)

print("Distribución binaria (0 = no rice, 1 = rice):")
print(y.value_counts().sort_index().rename(index={0: "no rice", 1: "rice"}))
print(f"Proporción positiva: {y.mean():.3%}")
print(f"Tamaños: train={len(y_train)}, validación={len(y_val)}, test={len(y_test)}")

## Etapa 2: Modelo base con CV

Entrenamos un Random Forest con hiperparámetros por defecto y evaluamos con CV 5-fold estratificado. La estratificación procura conservar la proporción de ejemplos de arroz en cada pliegue; aquí solo usamos el conjunto de entrenamiento.

In [ ]:
def costo_total(y_true, y_pred, C_FP, C_FN):
    """Calcula el costo económico total a partir de falsos positivos y negativos."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return int(fp * C_FP + fn * C_FN)


def evaluar_modelo_cv(modelo, X, y, cv=5):
    """Devuelve F1 medio, desviación estándar de F1 y costo pooled out-of-fold."""
    splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    f1_scores = cross_val_score(modelo, X, y, cv=splitter, scoring="f1", n_jobs=-1)
    pred_oof = cross_val_predict(modelo, X, y, cv=splitter, n_jobs=-1)
    costo_oof = costo_total(y, pred_oof, C_FP, C_FN)
    return float(f1_scores.mean()), float(f1_scores.std()), costo_oof


# Usamos el Random Forest por defecto como referencia reproducible.
modelo_base = RandomForestClassifier(random_state=SEED)
f1_base_cv, f1_base_std, costo_base_cv = evaluar_modelo_cv(
    modelo_base, X_train, y_train, cv=5
)
print(f"F1 CV: {f1_base_cv:.4f} ± {f1_base_std:.4f}")
print(f"Costo OOF CV (umbral 0.5): ${costo_base_cv:,.0f}")

# Guardamos métricas y artefactos comparables de cada variante.
resultados = {}

## Etapa 3: Matriz de costos

Calculamos el costo total del modelo base en validación. Recordar: **costo = FP × 50 + FN × 500.**

La matriz de confusión hace explícito cuántos errores de cada tipo se produjeron. El conjunto de prueba permanece reservado para la evaluación final.

In [ ]:
# Ajustamos la referencia y medimos su costo con la regla predeterminada de 0.5.
modelo_base.fit(X_train, y_train)
y_val_base = modelo_base.predict(X_val)
cm_base = confusion_matrix(y_val, y_val_base, labels=[0, 1])
tn, fp, fn, tp = cm_base.ravel()
costo_base_val = costo_total(y_val, y_val_base, C_FP, C_FN)
print("Matriz de confusión (filas=real, columnas=predicho):")
print(cm_base)
print(f"FP={fp}, FN={fn}, costo de validación=${costo_base_val:,.0f}")

# El costo de validación se usa para la comparación de técnicas.
resultados["base"] = {
    "f1_cv": f1_base_cv,
    "cost": costo_base_val,
    "model": modelo_base,
    "threshold": 0.5,
}

## Etapa 4: Threshold Tuning

Encontramos el umbral que minimiza el costo entre 0.1 y 0.9. El umbral se elige con el conjunto de **validación**, no con test: así evitamos optimizar directamente sobre los datos que usaremos como estimación final.

In [ ]:
def encontrar_umbral_optimo(modelo, X_test, y_test, C_FP, C_FN):
    """Busca entre 0.1 y 0.9 el umbral con menor costo en el conjunto recibido."""
    probabilidades = modelo.predict_proba(X_test)[:, 1]
    umbrales = np.arange(0.1, 1.0, 0.1)
    costos = [
        costo_total(y_test, (probabilidades >= umbral).astype(int), C_FP, C_FN)
        for umbral in umbrales
    ]
    indice_optimo = int(np.argmin(costos))
    return float(umbrales[indice_optimo]), int(costos[indice_optimo])


# Los nombres de parámetros de la función son genéricos; la llamada usa validación.
umbral_base, costo_threshold_val = encontrar_umbral_optimo(
    modelo_base, X_val, y_val, C_FP, C_FN
)
f1_threshold_val = f1_score(
    y_val, (modelo_base.predict_proba(X_val)[:, 1] >= umbral_base).astype(int)
)
print(f"Umbral óptimo en validación: {umbral_base:.1f}")
print(f"Costo mínimo en validación: ${costo_threshold_val:,.0f}")
resultados["threshold"] = {
    "f1_cv": f1_base_cv,
    "cost": costo_threshold_val,
    "model": modelo_base,
    "threshold": umbral_base,
}

## Etapa 5: `class_weight='balanced'`

Aplicamos `class_weight` para compensar el desbalance durante el entrenamiento. A diferencia del threshold tuning, esta técnica cambia la importancia de los errores en la función de aprendizaje; luego se calibra su umbral con validación.

In [ ]:
# Entrenamos con pesos inversamente proporcionales a las frecuencias de clase.
modelo_weighted = RandomForestClassifier(
    class_weight="balanced", random_state=SEED
)
f1_weighted_cv, f1_weighted_std, costo_weighted_cv = evaluar_modelo_cv(
    modelo_weighted, X_train, y_train, cv=5
)
modelo_weighted.fit(X_train, y_train)
umbral_weighted, costo_weighted_val = encontrar_umbral_optimo(
    modelo_weighted, X_val, y_val, C_FP, C_FN
)
print(f"F1 CV: {f1_weighted_cv:.4f} ± {f1_weighted_std:.4f}")
print(f"Costo OOF CV: ${costo_weighted_cv:,.0f}")
print(f"Umbral validación: {umbral_weighted:.1f}; costo: ${costo_weighted_val:,.0f}")
resultados["class_weight"] = {
    "f1_cv": f1_weighted_cv,
    "cost": costo_weighted_val,
    "model": modelo_weighted,
    "threshold": umbral_weighted,
}

## Etapa 6: Feature engineering

Creamos nuevas features: **ratios N/P, N/K, humidity/temperature, pH*rainfall**. Las razones pueden capturar proporciones de nutrientes y condiciones ambientales que resultan informativas para distinguir cultivos. Sumamos una pequeña constante al denominador para evitar divisiones por cero.

In [ ]:
def crear_features(df):
    """Agrega cuatro razones/interacciones agronómicas a una copia del DataFrame."""
    columnas = {"N", "P", "K", "humidity", "temperature", "ph", "rainfall"}
    faltantes = columnas.difference(df.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas para crear features: {sorted(faltantes)}")
    resultado = df.copy()
    epsilon = 1e-8
    resultado["N_P_ratio"] = resultado["N"] / (resultado["P"] + epsilon)
    resultado["N_K_ratio"] = resultado["N"] / (resultado["K"] + epsilon)
    resultado["hum_temp_ratio"] = resultado["humidity"] / (
        resultado["temperature"] + epsilon
    )
    resultado["ph_rainfall"] = resultado["ph"] * resultado["rainfall"]
    return resultado


# Construimos variables a partir de los datos originales y volvemos a escalar sin fuga.
X_train_eng_raw = crear_features(X_train_raw)
X_val_eng_raw = crear_features(X_val_raw)
X_test_eng_raw = crear_features(X_test_raw)
scaler_eng = RobustScaler()
X_train_eng = pd.DataFrame(
    scaler_eng.fit_transform(X_train_eng_raw),
    columns=X_train_eng_raw.columns,
    index=X_train_eng_raw.index,
)
X_val_eng = pd.DataFrame(
    scaler_eng.transform(X_val_eng_raw),
    columns=X_val_eng_raw.columns,
    index=X_val_eng_raw.index,
)
X_test_eng = pd.DataFrame(
    scaler_eng.transform(X_test_eng_raw),
    columns=X_test_eng_raw.columns,
    index=X_test_eng_raw.index,
)

# Comparamos el modelo ponderado con CV y elegimos el umbral en validación.
modelo_eng = RandomForestClassifier(class_weight="balanced", random_state=SEED)
f1_eng_cv, f1_eng_std, costo_eng_cv = evaluar_modelo_cv(
    modelo_eng, X_train_eng, y_train, cv=5
)
modelo_eng.fit(X_train_eng, y_train)
umbral_eng, costo_eng_val = encontrar_umbral_optimo(
    modelo_eng, X_val_eng, y_val, C_FP, C_FN
)
print(f"F1 CV: {f1_eng_cv:.4f} ± {f1_eng_std:.4f}")
print(f"Costo OOF CV: ${costo_eng_cv:,.0f}")
print(f"Costo en validación: ${costo_eng_val:,.0f}")
resultados["feature_eng"] = {
    "f1_cv": f1_eng_cv,
    "cost": costo_eng_val,
    "model": modelo_eng,
    "threshold": umbral_eng,
}

## Etapa 7: Feature selection

Seleccionamos las **top 5 features** más importantes del Random Forest entrenado con las variables de entrenamiento. La importancia de árboles es una señal exploratoria, no una prueba causal; puede cambiar entre muestras y favorecer variables con más puntos de corte.

In [ ]:
# Seleccionamos atributos usando únicamente importancias calculadas con entrenamiento.
modelo_importancias = RandomForestClassifier(
    class_weight="balanced", random_state=SEED
)
modelo_importancias.fit(X_train_eng, y_train)
importancias = pd.Series(
    modelo_importancias.feature_importances_, index=X_train_eng.columns
).sort_values(ascending=False)
features_top5 = importancias.head(5).index.tolist()
print("Top 5 features:")
print(importancias.head(5))

# Evaluamos solo el subconjunto seleccionado y mantenemos validación fuera del ajuste.
modelo_sel = RandomForestClassifier(class_weight="balanced", random_state=SEED)
f1_sel_cv, f1_sel_std, costo_sel_cv = evaluar_modelo_cv(
    modelo_sel, X_train_eng[features_top5], y_train, cv=5
)
modelo_sel.fit(X_train_eng[features_top5], y_train)
umbral_sel, costo_sel_val = encontrar_umbral_optimo(
    modelo_sel, X_val_eng[features_top5], y_val, C_FP, C_FN
)
print(f"F1 CV: {f1_sel_cv:.4f} ± {f1_sel_std:.4f}")
print(f"Costo OOF CV: ${costo_sel_cv:,.0f}; costo validación: ${costo_sel_val:,.0f}")
resultados["feature_sel"] = {
    "f1_cv": f1_sel_cv,
    "cost": costo_sel_val,
    "model": modelo_sel,
    "threshold": umbral_sel,
    "features": features_top5,
}

## Etapa 8: Optuna

Optimizamos hiperparámetros minimizando costo con validación cruzada de 5 pliegues. En esta función objetivo se aplica el umbral estándar 0.5; después de elegir los hiperparámetros, ajustaremos el umbral en validación. Así no mezclamos la búsqueda de parámetros con el conjunto final de prueba.

In [ ]:
def objective(trial):
    """Minimiza el costo pooled out-of-fold de un Random Forest en CV estratificada."""
    parametros = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=50),
        "max_depth": trial.suggest_categorical(
            "max_depth", [None, 5, 10, 20, 30]
        ),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "class_weight": trial.suggest_categorical(
            "class_weight", [None, "balanced"]
        ),
    }
    estimador = RandomForestClassifier(random_state=SEED, **parametros)
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    pred_oof = cross_val_predict(
        estimador, X_train, y_train, cv=splitter, n_jobs=-1
    )
    return costo_total(y_train, pred_oof, C_FP, C_FN)


# Ejecutamos una búsqueda acotada para mantener manejable el tiempo de clase.
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)
print("Mejores parámetros:", study.best_params)
print(f"Mejor costo OOF CV: ${study.best_value:,.0f}")

# Calibramos el threshold de la mejor configuración usando solo validación.
modelo_optuna = RandomForestClassifier(
    random_state=SEED, **study.best_params
)
f1_optuna_cv, f1_optuna_std, costo_optuna_cv = evaluar_modelo_cv(
    modelo_optuna, X_train, y_train, cv=5
)
modelo_optuna.fit(X_train, y_train)
umbral_optuna, costo_optuna_val = encontrar_umbral_optimo(
    modelo_optuna, X_val, y_val, C_FP, C_FN
)
print(f"F1 CV: {f1_optuna_cv:.4f} ± {f1_optuna_std:.4f}")
print(f"Costo con threshold ajustado en validación: ${costo_optuna_val:,.0f}")
resultados["optuna"] = {
    "f1_cv": f1_optuna_cv,
    "cost": costo_optuna_val,
    "model": modelo_optuna,
    "threshold": umbral_optuna,
}

## Etapa 9: SMOTE (opcional)

SMOTE genera muestras sintéticas de la clase minoritaria. Solo si el desbalance es severo y hay suficientes ejemplos representativos. Para CV, el remuestreo debe ocurrir **dentro de cada pliegue de entrenamiento**; por eso SMOTE se incluye en un `imblearn.Pipeline`. Validación y prueba conservan su distribución original.

In [ ]:
# Si imbalanced-learn no está instalado, omitimos esta comparación sin detener el notebook.
if SMOTE is None:
    print("SMOTE omitido: instale con %pip install imbalanced-learn para habilitarlo.")
    resultados["smote"] = {
        "f1_cv": np.nan,
        "cost": np.nan,
        "model": None,
        "threshold": np.nan,
    }
else:
    # SMOTE se ajusta solo en el entrenamiento de cada fold y del modelo final.
    modelo_smote = ImbPipeline(
        steps=[
            ("smote", SMOTE(random_state=SEED)),
            (
                "model",
                RandomForestClassifier(
                    class_weight="balanced", random_state=SEED
                ),
            ),
        ]
    )
    f1_smote_cv, f1_smote_std, costo_smote_cv = evaluar_modelo_cv(
        modelo_smote, X_train, y_train, cv=5
    )
    modelo_smote.fit(X_train, y_train)
    umbral_smote, costo_smote_val = encontrar_umbral_optimo(
        modelo_smote, X_val, y_val, C_FP, C_FN
    )
    print(f"F1 CV: {f1_smote_cv:.4f} ± {f1_smote_std:.4f}")
    print(f"Costo OOF CV: ${costo_smote_cv:,.0f}")
    print(f"Costo en validación: ${costo_smote_val:,.0f}")
    resultados["smote"] = {
        "f1_cv": f1_smote_cv,
        "cost": costo_smote_val,
        "model": modelo_smote,
        "threshold": umbral_smote,
    }

## Tabla comparativa final

Comparamos las siete variantes disponibles. El costo de esta tabla se calcula en **validación** (para el modelo base se usa el umbral 0.5); así las técnicas comparten el mismo conjunto de comparación. `ahorro_vs_base` es una diferencia de costos positiva cuando hay ahorro. La evaluación final en test se reserva para el modelo ya elegido.

In [ ]:
# Consolidamos los resultados y dejamos como NaN una técnica opcional no ejecutada.
tecnicas = [
    ("base", "Base"),
    ("threshold", "Threshold"),
    ("class_weight", "Class weight"),
    ("feature_eng", "Feature engineering"),
    ("feature_sel", "Feature selection"),
    ("optuna", "Optuna"),
    ("smote", "SMOTE"),
]
costo_base = resultados["base"]["cost"]
comparacion = pd.DataFrame(
    [
        {
            "técnica": etiqueta,
            "F1_CV": resultados[clave]["f1_cv"],
            "costo": resultados[clave]["cost"],
            "ahorro_vs_base": (
                costo_base - resultados[clave]["cost"]
                if pd.notna(resultados[clave]["cost"])
                else np.nan
            ),
        }
        for clave, etiqueta in tecnicas
    ]
)
comparacion.to_csv(REPORTS_DIR / "pipeline_b2.csv", index=False)
print(comparacion.to_string(index=False))
print(f"Reporte guardado en: {REPORTS_DIR / 'pipeline_b2.csv'}")

## Gráfico comparativo

El gráfico resume los costos de validación de cada técnica; una barra más corta representa menor costo bajo los costos asumidos. Si SMOTE no estaba disponible, su valor faltante se omite del gráfico.

In [ ]:
# Graficamos y guardamos únicamente técnicas con costo disponible.
datos_grafico = comparacion.dropna(subset=["costo"])
ax = sns.barplot(data=datos_grafico, x="técnica", y="costo", color="steelblue")
ax.set_title("Costo de validación por técnica")
ax.set_xlabel("Técnica")
ax.set_ylabel("Costo total ($)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pipeline_b2.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Gráfico guardado en: {FIGURES_DIR / 'pipeline_b2.png'}")

## Entrenar modelo final

Entrenamos el modelo final con la mejor combinación de Optuna y el umbral óptimo elegido en validación. Solo ahora consultamos el conjunto de test, una vez, para estimar cómo podría generalizar la decisión fijada.

In [ ]:
# Reentrenamos la configuración ganadora solo con la partición de ajuste.
modelo_final = RandomForestClassifier(
    **study.best_params, random_state=SEED
)
modelo_final.fit(X_train, y_train)
umbral_final, costo_val_final = encontrar_umbral_optimo(
    modelo_final, X_val, y_val, C_FP, C_FN
)

# Medimos una sola vez en test, sin cambiar hiperparámetros ni umbral.
prob_test = modelo_final.predict_proba(X_test)[:, 1]
y_pred_test = (prob_test >= umbral_final).astype(int)
costo_test_final = costo_total(y_test, y_pred_test, C_FP, C_FN)
f1_test_final = f1_score(y_test, y_pred_test, zero_division=0)
cm_test_final = confusion_matrix(y_test, y_pred_test, labels=[0, 1])
print(f"Umbral elegido en validación: {umbral_final:.1f}")
print(f"Costo de validación: ${costo_val_final:,.0f}")
print(f"Costo final en test: ${costo_test_final:,.0f}")
print(f"F1 final en test: {f1_test_final:.4f}")
print("Matriz de confusión en test:")
print(cm_test_final)

# Persistimos el estimador y la configuración necesaria para reproducir la decisión.
model_path = MODELS_DIR / "rf_final_b2.pkl"
joblib.dump(modelo_final, model_path)
config = {
    "best_params": study.best_params,
    "threshold": float(umbral_final),
    "C_FP": int(C_FP),
    "C_FN": int(C_FN),
    "seed": int(SEED),
    "feature_columns": list(X_train.columns),
    "validation_cost": int(costo_val_final),
    "test_cost": int(costo_test_final),
    "test_f1": float(f1_test_final),
    "model_path": str(model_path.relative_to(PROJECT_ROOT)),
}
with (PROJECT_ROOT / "config.json").open("w", encoding="utf-8") as archivo:
    json.dump(config, archivo, indent=2, ensure_ascii=False)
print(f"Modelo guardado en: {model_path}")
print(f"Configuración guardada en: {PROJECT_ROOT / 'config.json'}")

## Criterios para producción

Antes de desplegar, revisa cada punto:

- [ ] El costo esperado es menor que el beneficio del caso de uso.
- [ ] El rendimiento es estable entre particiones y periodos.
- [ ] Los resultados son reproducibles con datos, código y semilla registrados.
- [ ] El pipeline y sus supuestos están documentados.
- [ ] Hay pruebas para entradas, salidas y casos límite.
- [ ] La latencia cumple los requisitos operativos.
- [ ] El comportamiento es suficientemente interpretable para quienes toman decisiones.

**Si algún ítem falla, el modelo no está listo.** Un buen puntaje de validación, por sí solo, no prueba factibilidad operacional.

## Conexión con Bloque 3

| Bloque 2 | Bloque 3 |
|---|---|
| Modelo optimizado, costo esperado y umbral de decisión | Exportación a ONNX, servicio con FastAPI, empaquetado con Docker |
| Decidir si el modelo vale la pena | Profiling de recursos y latencia; Green AI y sostenibilidad |

**B2 respondió: ¿vale la pena? B3 responderá: ¿cómo lo despliego?**

## Conclusiones

1. Un pipeline de siete etapas conecta problema, costos, evaluación, optimización y documentación.
2. Threshold tuning, pesos de clase, variables derivadas, selección, SMOTE y Optuna ofrecen palancas distintas.
3. Conviene seguir el orden recomendado y validar cada cambio con CV, sin reutilizar test para tomar decisiones.
4. La producción exige beneficio, estabilidad, reproducibilidad, documentación, pruebas, latencia e interpretabilidad.
5. El Bloque 2 decide si vale la pena; el Bloque 3 abordará despliegue, rendimiento y sostenibilidad.

**Próxima clase:** inicio del Bloque 3 (relevancia, sostenibilidad y proyectos).

In [ ]:
# Confirmamos el cierre y recordamos dónde consultar el reporte y la configuración.
print("Clase 13 completada. Bloque 2 finalizado. Revisa reports/pipeline_b2.csv y config.json")